# 01 — Corpus table

The starting point. One tidy table for the whole corpus, built from the 30
per-page JSON files in `data/processed/`.

**One row = one article** (a headline + its body). Metadata like the era and
page number ride along as columns.

This table is what everything downstream vectorises: TF-IDF, LDA and embeddings
all take the `body` column as their input. So it's worth getting clean here once.


In [1]:
import json
from pathlib import Path

import pandas as pd

PROCESSED = Path("../data/processed")
files = sorted(PROCESSED.glob("*.json"))
len(files)  # expect 30

30

## Flatten JSON → rows

Each JSON file is one page and holds a *list* of articles. We want to unroll
that: walk every file, walk every article inside it, and emit one dict per
article carrying the page-level metadata (era, page number) down onto it.


In [2]:
def load_articles(files):
    """Return a list of per-article dicts, flattened from the page JSON files.

    Each output dict should have at least:
        source_file, page_number, era_label, headline, body

    Steps:
      1. loop over files
      2. json.load each one  -> a page dict
      3. loop over page["articles"]
      4. build a row dict pulling era_label / page_number from the PAGE
         and headline / body from the ARTICLE
      5. collect and return all the rows
    """
    rows = []
    # TODO: your loop here
    for article_file in files:
        with open(article_file, "r") as f:
            page = json.load(f)
            for article in page["articles"]:
                row = {
                    "source_file": article_file.name,
                    "page_number": page.get("page_number"),
                    "era_label": page.get("era_label"),
                    "headline": article.get("headline"),
                    "body": article.get("body"),
                }
                rows.append(row)
    return rows

rows = load_articles(files)
len(rows)  # expect 81

81

## Into a DataFrame

Once `rows` is a list of flat dicts, this is one line. Then add a word-count
column — you'll use it constantly for sanity checks (empty bodies, outliers).


In [3]:
df = pd.DataFrame(rows)

df["n_words"] = df["body"].str.split().str.len()

df.head()

,source_file,page_number,era_label,headline,body,n_words
0,Pages 30-37_p01.json,30,1911-12,Battling Barnsley show Yorkshire grit,BARNSLEY played six goalless games in an extra...,255
1,Pages 30-37_p01.json,30,1911-12,Another Olympic triumph for England,ENGLAND represented the United Kingdom again a...,238
2,Pages 30-37_p02.json,31,1912-13,Battle of the giants ends level,THE BATTLE of the giants has ended with honour...,521
3,Pages 30-37_p02.json,31,1912-13,Ten Irish heroes,ENGLAND LOST to Ireland for the first time in ...,143
4,Pages 30-37_p03.json,32,1913-14,A great day for the Irish,IRELAND have won the International Championshi...,304


## Sanity checks

Before trusting the table, look at it. Questions to answer with a line each:

- How many rows? (should be 81)
    - It is 81
- Any empty bodies left? (`n_words == 0`)
    - No empty bodies
- Articles per era — does it match what you saw in the book?
    - For the most part yes
- What do the longest and shortest articles look like?
    - Largest are long, smallest has some issues. I see a NaN in headline because it was part of a football feature

In [4]:
df.shape
df[df["n_words"] == 0]
df["era_label"].value_counts()
df.nlargest(3, "n_words")[["era_label", "headline", "n_words"]]
df.nsmallest(3, "n_words")[["page_number","era_label", "headline", "n_words"]]

,page_number,era_label,headline,n_words
64,12,THE 1890s,No game for ladies!,47
73,15,FOOTBALL FEATURE,NaN,68
42,27,FOOTBALL FEATURE,Sunderland talents,77


## Save

Persist the clean table so the FE notebooks (`02_tfidf`, `03_lda`, …) just load
this instead of re-parsing JSON. Parquet keeps dtypes; CSV is fine too.


In [5]:
df.to_parquet("../data/corpus.parquet")